In [30]:
from copy import deepcopy
from graphviz import Digraph

class Node:
  def __init__(self, state, action = None, parent = None):
    self.state = state # 2D list (3x3)
    self.id = str(self) # identifier of node
    self.action = action
    self.parent = parent

  def __str__(self):
    rs=''
    for i in range(3):
      for j in range(3):
        if(self.state[i][j]==0):
          rs+='_'
          continue
        rs+=str(self.state[i][j])
      rs+='\n'
    return rs

  def get_successors(self):
    successors = []
    actions = ['Left', 'Right', 'Up', 'Down']
    for action in actions:
        new_state = deepcopy(self.state)
        successor_state = self.get_successor(action, new_state)
        if successor_state is not None:
          successor_node = Node(successor_state, action, self)
          successors.append(successor_node)
    return successors

  def get_successor(self, action, state):
    pi, pj = self.get_blank_pos(state)
    pi, pj = self.get_dest_pos(action, pi, pj)
    if 0 <= pi and pi < 3 and 0 <= pj and pj < 3:
      if action == 'Left':
        state[pi][pj - 1] = state[pi][pj]
      if action == 'Right':
        state[pi][pj + 1] = state[pi][pj]
      if action == 'Up':
        state[pi-1][pj] = state[pi][pj]
      if action == 'Down':
        state[pi+1][pj] = state[pi][pj]
      state[pi][pj] = 0
      return state
    return None

  def get_dest_pos(self, action, pi, pj):
    if action == 'Left':
      pj += 1
    if action == 'Right':
      pj -= 1
    if action == 'Up':
      pi += 1
    if action == 'Down':
      pi -= 1
    return pi, pj

  def get_blank_pos(self,state):
    for i in range(3):
      for j in range(3):
        if(state[i][j]==0):
          return i,j


  def get_id(self):
    return self.id

  def get_node_str(self):
    return str(self)

  def get_action(self):
    return self.action

  def draw(self, dot):
    dot.node(self.get_id(), self.get_node_str())
    if self.parent is not None:
      dot.edge(self.parent.get_id(), self.get_id(), self.get_action())
  def __eq__(self,other):
    for i in range(3):
      for j in range(3):
        if(self.state[i][j]!=other[i][j]):
          return False
        else:
          continue
    return True

In [31]:
import heapq
class PriorityQueue:
    def  __init__(self):
        self.heap = []
        self.count = 0

    def push(self, item, priority):
        entry = (priority, self.count, item)
        heapq.heappush(self.heap, entry)
        self.count += 1

    def pop(self):
        (_, _, item) = heapq.heappop(self.heap)
        return item

    def isEmpty(self):
        return len(self.heap) == 0

    def update(self, item, priority):
        for index, (p, c, i) in enumerate(self.heap):
            if i == item:
                if p <= priority:
                    break
                del self.heap[index]
                self.heap.append((priority, c, item))
                heapq.heapify(self.heap)
                break
        else:
            self.push(item, priority)
class Queue:
    def __init__(self):
        self.list = []

    def push(self,item):
        self.list.insert(0,item)

    def pop(self):
        return self.list.pop()

    def isEmpty(self):
        return len(self.list) == 0

In [ ]:

def manhattanHeuristic(current, goal_state):
    rs=0
    for i in range(3):
      for j in range (3):
        if(current[i][j!=0]):
          h,k=getGoalStatePos(current[i][j],goal_state)
          rs+=abs(i - h) + abs(j - k)
    return rs

def euclideanHeuristic(current, goal_state):
    rs=0
    for i in range(3):
      for j in range (3):
        if(current[i][j!=0]):
          h,k=getGoalStatePos(current[i][j],goal_state)
          rs+=((i - h) ** 2 + (j - k) ** 2) ** 0.5
    return rs
def getGoalStatePos(n:int, goal_state):
  for i in range(3):
    for j in range(3):
      if goal_state[i][j]==n:
        return i,j

def aStarSearch(node:Node,heuristic=manhattanHeuristic):
  visited=set()
  statePQ=PriorityQueue()
  goal_state1=[[1,2,3],[4,5,6],[7,8,0]]
  goal_state2=[[0,1,2],[3,4,5],[6,7,8]]
  heuristicValue1=heuristic(node.state,goal_state1)
  heuristicValue2=heuristic(node.state,goal_state2)
  current=node
  statePQ.push((node,[],0),heuristicValue1)
  statePQ.push((node,[],0),heuristicValue2)
  dot=Digraph()
  i=0
  while not statePQ.isEmpty():
    current,cActions,cCost=statePQ.pop()
    if(current.__eq__(goal_state1) or current.__eq__(goal_state2)):
      current.draw(dot)
      break
    if(current.get_id() in visited):
      continue
    visited.add(current.get_id())
    current.draw(dot)
    for successor in current.get_successors():
      sState=successor.state
      nextAction=successor.action
      sActions=cActions+[nextAction]
      sCost=cCost+1
      heuristicValue1=heuristic(sState,goal_state1)+sCost
      heuristicValue2=heuristic(sState,goal_state2)+sCost
      print(successor.get_id())
      print(heuristicValue1,heuristicValue2)
      statePQ.push((successor,sActions,sCost),heuristicValue1)
      statePQ.push((successor,sActions,sCost),heuristicValue2)
  while not (node.__eq__(goal_state1) or node.__eq__(goal_state2)):
      for successor in node.get_successors():
        if successor.action== cActions[i]:
          i+=1
          successor.draw(dot)
          node=successor
          break
  return dot,cCost,cActions


In [33]:
def breadthFirstSearch(node):
    visited = set()
    stateStack = Queue()
    stateStack.push((node, [], 0))
    goal_state1=[[1,2,3],[4,5,6],[7,8,0]]
    goal_state2=[[0,1,2],[3,4,5],[6,7,8]]
    dot=Digraph()
    i=0
    while not stateStack.isEmpty():
        current, cActions, cCost = stateStack.pop()

        if current.__eq__(goal_state1) or current.__eq__(goal_state2):
            break

        if (current.get_id() in visited):
            continue

        visited.add(current.get_id())
        for successor in current.get_successors():
            nextAction= successor.action

            sActions = cActions + [nextAction]
            sCost = cCost + 1
            if successor.get_id() not in visited:
              stateStack.push((successor, sActions, sCost))
    while not (node.__eq__(goal_state1) or node.__eq__(goal_state2)) :
      for successor in node.get_successors():
        if successor.action== cActions[i]:
          i+=1
          successor.draw(dot)
          node=successor
          break
    return dot,cCost,cActions

In [34]:
from graphviz import Digraph
from numpy import *
def isLegal(A):
  count=0
  for i in range(8):
    for j in range(i+1,9):
      if(A[i]==0):
        continue
      if(A[i]>A[j] and A[j]!=0):
        count+=1
  rs= count%2==0
  return rs
def randomNode():
  A=[0,1,2,3,4,5,6,7,8]
  random.shuffle(A)
  while not isLegal(A):
    random.shuffle(A)
  n=Node(array(A).reshape(3,3))
  return n


In [ ]:

if __name__ == '__main__':
  print("Nhap day so co format nhu sau: 0 1 2 3 4 5 6 7 8")
  A= input().split()
  A= [int(a) for a in A]
  check=True
  for a in A:
    if a<0 or a>8:
      check=False
      break
  while len(A)!=len(set(A)) or len(set(A))!=9 or not isLegal(A) or check==False :
    print("Vui long nhap lai day so khac")
    A= input().split()
    A= [int(a) for a in A]
    check=True
    for a in A:
      if a<0 or a>8:
        check=False
        break
  n=Node(array(A).reshape(3,3))
  print("Ghi ten giai thuat: breadthFirstSearch/aStarSearch ")
  choose= input()
  if(choose=='breadthFirstSearch'):
    dotRs,totalcost,path=breadthFirstSearch(n)
  elif(choose=='aStarSearch'):
    dotRs,totalcost,path=aStarSearch(n,heuristic=manhattanHeuristic)
  else:
    print('Failure')
    dotRs,totalcost,path=None,0,None
  print("Path: ",path)
  print("Total Cost: ",totalcost)
dotRs

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

def compare_algorithms_performance(number_of_trials):
    bfs_times = []
    a_star_times = []
    bfs_costs = []
    a_star_costs = []

    for _ in range(number_of_trials):
        n = randomNode()  # Generate a random legal node
        # Measure BFS performance
        start_time = time.time()
        _, costBFS, _ = breadthFirstSearch(n)
        end_time = time.time()
        bfs_times.append(end_time - start_time)
        bfs_costs.append(costBFS)

        # Measure A* performance
        start_time = time.time()
        _, costAs, _ = aStarSearch(n, heuristic=euclideanHeuristic)
        end_time = time.time()
        a_star_times.append(end_time - start_time)
        a_star_costs.append(costAs)

    # Calculating average time and costs
    avg_bfs_time = np.mean(bfs_times)
    avg_a_star_time = np.mean(a_star_times)
    avg_bfs_cost = np.mean(bfs_costs)
    avg_a_star_cost = np.mean(a_star_costs)

    # Drawing the bar graphs
    labels = ['BFS', 'A*']
    time_values = [avg_bfs_time, avg_a_star_time]
    cost_values = [avg_bfs_cost, avg_a_star_cost]

    x = np.arange(len(labels))  # the label locations
    width = 0.35  # the width of the bars

    fig, ax = plt.subplots()
    rects1 = ax.bar(x - width/2, time_values, width, label='Time')
    rects2 = ax.bar(x + width/2, cost_values, width, label='Cost')

    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel('Scores')
    ax.set_title('Performance comparison of BFS and A* in %d times'%(number_of_trials))
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    ax.bar_label(rects1, padding=3)
    ax.bar_label(rects2, padding=3)

    fig.tight_layout()

    plt.show()

# Call the function to compare the performance
compare_algorithms_performance(1000)
